In [39]:
df = pd.read_csv('twitter_training.csv')
df.head()

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [40]:
columns = ['Tweet ID','Entity', 'Sentiment', 'Tweet Content']
df.columns = columns
df.head()

,Tweet ID,Entity,Sentiment,Tweet Content
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [41]:
df['Tweet Content']

0        I am coming to the borders and I will kill you...
1        im getting on borderlands and i will kill you ...
2        im coming on borderlands and i will murder you...
3        im getting on borderlands 2 and i will murder ...
4        im getting into borderlands and i can murder y...
                               ...                        
74676    Just realized that the Windows partition of my...
74677    Just realized that my Mac window partition is ...
74678    Just realized the windows partition of my Mac ...
74679    Just realized between the windows partition of...
74680    Just like the windows partition of my Mac is l...
Name: Tweet Content, Length: 74681, dtype: object

In [42]:
df['Sentiment'].unique()

array(['Positive', 'Neutral', 'Negative', 'Irrelevant'], dtype=object)

In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74681 entries, 0 to 74680
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Tweet ID       74681 non-null  int64 
 1   Entity         74681 non-null  object
 2   Sentiment      74681 non-null  object
 3   Tweet Content  73995 non-null  object
dtypes: int64(1), object(3)
memory usage: 2.3+ MB


In [44]:
df.isna().sum()

Tweet ID           0
Entity             0
Sentiment          0
Tweet Content    686
dtype: int64

In [45]:
#df.dropna(inplace=True)
#df.isna().sum()
df.fillna(df.mean(numeric_only= True), inplace=True)
df.fillna(df.mode().iloc[0], inplace= True)

In [46]:
x = df['Tweet Content']#.str.strip().str.lower()
y = df['Sentiment']#.str.strip().str.lower()

In [48]:
import re
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

def clean(text):
    text=str(text).lower()
    text=re.sub(r'@\w+', '', text)
    text=re.sub(r'\d+', '', text)
    text=re.sub(r'https?://\S+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    #text = re.sub(r'[!\"#$%&'()*+,\-./:;<=>?@\[\\\]^_`{|}~ ]', '', text)
    return text


x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state= 42)

tfidf = TfidfVectorizer(stop_words= 'english', preprocessor= clean, max_features = 10000, ngram_range = (1,2))
x_train_tfidf = tfidf.fit_transform(x_train)
x_test_tfidf = tfidf.transform(x_test)
bayes_model = MultinomialNB()
bayes = bayes_model.fit(x_train_tfidf, y_train)

y_pred = bayes.predict(x_test_tfidf)

from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
classification = classification_report(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)
confusion = confusion_matrix(y_test, y_pred)
print(confusion)
print(accuracy)
print(classification)

[[1304  848  331  806]
 [  88 4690  346  460]
 [ 170 1056 2455  776]
 [ 115  823  388 4015]]
0.6675593165872208
              precision    recall  f1-score   support

  Irrelevant       0.78      0.40      0.53      3289
    Negative       0.63      0.84      0.72      5584
     Neutral       0.70      0.55      0.62      4457
    Positive       0.66      0.75      0.70      5341

    accuracy                           0.67     18671
   macro avg       0.69      0.63      0.64     18671
weighted avg       0.68      0.67      0.66     18671



In [49]:
new = bayes.predict(tfidf.transform(['i dont know if i hate or love this product!']))
new

array(['Negative'], dtype='<U10')

WITH PIPELINE

In [70]:
import re
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

def clean(text):
    text=str(text).lower()
    text=re.sub(r'@\w+', '', text)
    text=re.sub(r'\d+', '', text)
    text=re.sub(r'https?://\S+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    #text = re.sub(r'[!\"#$%&'()*+,\-./:;<=>?@\[\\\]^_`{|}~ ]', '', text)
    return text


x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state= 42)

pipe = Pipeline(
    steps=[
        #('clean', FunctionTransformer(clean)),
        ('vecro', TfidfVectorizer(stop_words= 'english', preprocessor=clean, max_features = 10000, ngram_range = (1,2))),
        ('estimator', MultinomialNB())
    ]
)
pipe.fit(x_train, y_train)
pred = pipe.predict(x_test)

classification = classification_report(y_test, pred)
accuracy = accuracy_score(y_test, pred)
confusion = confusion_matrix(y_test, pred)
print(confusion)
print(accuracy)
print(classification)

new_ = pipe.predict(['i dont know if i hate or love this product!',
                     'i dont know if i hate  this product!'])
for i in new_:
    print(i)

[[1304  848  331  806]
 [  88 4690  346  460]
 [ 170 1056 2455  776]
 [ 115  823  388 4015]]
0.6675593165872208
              precision    recall  f1-score   support

  Irrelevant       0.78      0.40      0.53      3289
    Negative       0.63      0.84      0.72      5584
     Neutral       0.70      0.55      0.62      4457
    Positive       0.66      0.75      0.70      5341

    accuracy                           0.67     18671
   macro avg       0.69      0.63      0.64     18671
weighted avg       0.68      0.67      0.66     18671

Negative
Negative
